# Hindi No-Position Bonus: Evaluation Recovery
Attach the original bonus bundle and the saved output from the completed training run, then enable a GPU. **Save & Run All performs evaluation only; it cannot restart training.**

In [ ]:
from pathlib import Path
import shutil

input_root = Path('/kaggle/input')
manifests = list(input_root.rglob('bonus_bundle_manifest.json'))
if manifests:
    bundle = manifests[0].parent
else:
    archives = list(input_root.rglob('bonus_hindi_no_position_kaggle.zip'))
    assert len(archives) == 1, f'Expected one bundle, found: {archives}'
    extract_root = Path('/kaggle/working/bonus_bundle')
    shutil.unpack_archive(archives[0], extract_root)
    bundle = extract_root / 'lma_bonus_bundle'
assert (bundle / 'bonus_bundle_manifest.json').exists()
print('Bundle:', bundle)

In [ ]:
%pip install -q sentencepiece sacrebleu safetensors

In [ ]:
import json, os, subprocess, sys

latest_files = [p for p in Path('/kaggle/input').rglob('latest.json') if p.parent.name == 'run']
assert latest_files, 'Attach the saved output containing hindi_no_position/run/latest.json'
saved_run = latest_files[0].parent
run_dir = Path('/kaggle/working/hindi_no_position/run')
if not run_dir.exists():
    run_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(saved_run, run_dir)
latest = json.loads((run_dir / 'latest.json').read_text())
assert latest['training_step'] == 20300, latest
ablated_checkpoint = run_dir / latest['checkpoint']
assert ablated_checkpoint.exists(), ablated_checkpoint

repo = bundle / 'repo'
data = bundle / 'data/hindi'
test_jsonl = next((p for p in (data / 'test.jsonl.gz', data / 'test.jsonl') if p.exists()), None)
assert test_jsonl is not None, f'Missing test JSONL under {data}'
report = Path('/kaggle/working/hindi_no_position/report')
report.mkdir(parents=True, exist_ok=True)
environment = os.environ.copy()
environment['PYTHONPATH'] = str(repo / 'src')

def evaluate(name, config, checkpoint):
    command = [sys.executable, '-m', 'lma_phase2.evaluate',
        '--config', str(config), '--checkpoint', str(checkpoint),
        '--test-manifest', str(data / 'test.manifest.json'),
        '--test-jsonl', str(test_jsonl),
        '--tokenizer-model', str(repo / 'hindi/tokenizer/hindi.model'),
        '--output', str(report / f'{name}_evaluation.json'),
        '--samples-output', str(report / f'{name}_samples.jsonl'),
        '--attention-csv', str(report / f'{name}_attention.csv'),
        '--figure-dir', str(report / 'attention' / name), '--device', 'cuda']
    subprocess.run(command, check=True, cwd=repo, env=environment)

evaluate('baseline', repo / 'hindi/configs/phase2.toml',
         bundle / 'baseline/checkpoint_step_0020300.pt')
evaluate('no_position', repo / 'hindi/configs/phase2_no_position.toml',
         ablated_checkpoint)
subprocess.run([sys.executable, '-m', 'lma_phase2.plot_training',
    '--log', f'Hindi NoPE={run_dir / "training_log.csv"}',
    '--output', str(report / 'training_curve.png')],
    check=True, cwd=repo, env=environment)

In [ ]:
from IPython.display import display
import json

report = Path('/kaggle/working/hindi_no_position/report')
display(json.loads((report / 'baseline_evaluation.json').read_text()))
display(json.loads((report / 'no_position_evaluation.json').read_text()))
print('Save the complete /kaggle/working/hindi_no_position directory as notebook output.')